# ================================================================
#     VISIONFORGE: YOLO MASTER VIDEO COMPARISON & VERIFICATION
#       Compare Previous best vs New best.pt | Annotated Video
#                     Google Colab + Cloud GPU
# ================================================================
This notebook runs high-definition video inference using your newly trained YOLO `best.pt` checkpoint.

### Features:
- **Dual Model Benchmark**: Compare **Previous Model** vs **New `best.pt`** side-by-side.
- **Flexible Model Ingestion**: Auto-detect from Google Drive, direct `.pt` upload, or pretrained base weights.
- **Video Processing**: Upload `.mp4`, select from Google Drive, or auto-compile dataset validation frames.
- **Enterprise Telemetry HUD**: Real-time FPS, detection count, confidence distributions, and IoU overlap agreement.
- **In-Browser Playback & Download**: Watch the synchronized video directly in Colab and 1-click download.

In [ ]:
#@title 1. Install Dependencies & GPU Verification
import sys
import subprocess

print("Installing Ultralytics, OpenCV, and PyYAML...")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U", "ultralytics", "opencv-python", "PyYAML"
])

import torch
print("=" * 65)
print("         VISIONFORGE YOLO VIDEO COMPARISON ENGINE")
print("=" * 65)
if torch.cuda.is_available():
    print("[OK] GPU Device:  ", torch.cuda.get_device_name(0))
    print("[OK] CUDA Version:", torch.version.cuda)
    print("[OK] VRAM:        ", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")
else:
    print("[WARNING] GPU unavailable! For fast inference, go to:")
    print("          Runtime -> Change runtime type -> T4 GPU")


In [ ]:
#@title 2. Connect Google Drive
from pathlib import Path
from google.colab import drive

print("Mounting Google Drive...")
drive.mount("/content/drive")

BASE_DIR = Path("/content/yolo_verification")
DRIVE_ROOT = Path("/content/drive/MyDrive/YOLO_MASTER_TRAINING")
BASE_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

print("[OK] Google Drive connected.")
print("[OK] Checkpoints directory:", DRIVE_ROOT)


In [ ]:
#@title 3. Configure Model & Video Comparison Options
# Model & Execution Mode
VERIFICATION_MODE = "Dual Model Comparison (Previous vs New)" #@param ["Dual Model Comparison (Previous vs New)", "Single Model Video Annotation (New best.pt)"]

# New Model (Trained best.pt)
NEW_MODEL_SOURCE = "Auto-Detect from Drive (Latest Run)" #@param ["Auto-Detect from Drive (Latest Run)", "Upload from Computer (.pt file)", "Specify Path / URL"]
NEW_MODEL_PATH = "" #@param {type:"string"}

# Previous Model (Baseline or Old Checkpoint)
PREVIOUS_MODEL_SOURCE = "Pretrained Baseline (yolo26l.pt)" #@param ["Pretrained Baseline (yolo26l.pt)", "Pretrained Baseline (yolov8s.pt)", "Upload from Computer (.pt file)", "Select Earlier Drive Run", "Specify Path / URL"]
PREVIOUS_MODEL_PATH = "" #@param {type:"string"}

# Video Source
VIDEO_SOURCE = "Upload Video (.mp4) from Computer" #@param ["Upload Video (.mp4) from Computer", "Use Video from Google Drive", "Compile Dataset Validation Frames to Video"]
VIDEO_CUSTOM_PATH = "" #@param {type:"string"}

# Visual & Detection Settings
CONFIDENCE_THRESHOLD = 0.25 #@param {type:"number"}
IOU_MATCH_THRESHOLD = 0.50 #@param {type:"number"}
RENDER_MODE = "side_by_side" #@param ["side_by_side", "overlay"]
MAX_FRAMES_TO_PROCESS = 0 #@param {type:"integer"}

print(f"Configured Mode: {VERIFICATION_MODE}")
print(f"Render Style:    {RENDER_MODE} (Conf: {CONFIDENCE_THRESHOLD}, IoU: {IOU_MATCH_THRESHOLD})")


In [ ]:
#@title 4. Load Models & Ingest Video
import os
import cv2
from pathlib import Path
from google.colab import files

# --- 1. RESOLVE NEW MODEL (best.pt) ---
resolved_new_model = None
if NEW_MODEL_SOURCE == "Auto-Detect from Drive (Latest Run)":
    drive_pts = sorted(DRIVE_ROOT.glob("**/best.pt"), key=os.path.getmtime, reverse=True)
    if drive_pts:
        resolved_new_model = str(drive_pts[0])
        print("[FOUND] Latest trained best.pt in Drive:", resolved_new_model)
    else:
        # Check local runs folder
        local_pts = sorted(Path("/content/yolo_master/runs").glob("**/best.pt"), key=os.path.getmtime, reverse=True)
        if local_pts:
            resolved_new_model = str(local_pts[0])
            print("[FOUND] Local best.pt in /content/yolo_master/runs:", resolved_new_model)
        else:
            print("[NOTICE] No best.pt found in Drive or local runs. Please upload below:")
            up = files.upload()
            resolved_new_model = f"/content/{list(up.keys())[0]}"
elif NEW_MODEL_SOURCE == "Upload from Computer (.pt file)":
    print("Please upload your NEW best.pt file from your computer:")
    up = files.upload()
    if not up:
        raise ValueError("No file uploaded.")
    resolved_new_model = f"/content/{list(up.keys())[0]}"
else:
    resolved_new_model = NEW_MODEL_PATH.strip()

print("[OK] Resolved New Model:", resolved_new_model)

# --- 2. RESOLVE PREVIOUS MODEL ---
resolved_prev_model = None
if "Dual" in VERIFICATION_MODE:
    if "yolo26l.pt" in PREVIOUS_MODEL_SOURCE:
        resolved_prev_model = "yolo26l.pt"
    elif "yolov8s.pt" in PREVIOUS_MODEL_SOURCE:
        resolved_prev_model = "yolov8s.pt"
    elif PREVIOUS_MODEL_SOURCE == "Upload from Computer (.pt file)":
        print("Please upload your PREVIOUS model (.pt) from your computer:")
        up = files.upload()
        if not up:
            raise ValueError("No previous model uploaded.")
        resolved_prev_model = f"/content/{list(up.keys())[0]}"
    elif PREVIOUS_MODEL_SOURCE == "Select Earlier Drive Run":
        drive_pts = sorted(DRIVE_ROOT.glob("**/best.pt"), key=os.path.getmtime, reverse=True)
        if len(drive_pts) > 1:
            resolved_prev_model = str(drive_pts[1])
        elif drive_pts:
            resolved_prev_model = str(drive_pts[0])
        else:
            resolved_prev_model = "yolo26l.pt"
    else:
        resolved_prev_model = PREVIOUS_MODEL_PATH.strip() or "yolo26l.pt"
    print("[OK] Resolved Previous Model:", resolved_prev_model)

# --- 3. RESOLVE VIDEO SOURCE ---
resolved_video = None
if VIDEO_SOURCE == "Upload Video (.mp4) from Computer":
    print("Please upload your test video (.mp4) from your computer:")
    up = files.upload()
    if not up:
        raise ValueError("No video uploaded.")
    resolved_video = f"/content/{list(up.keys())[0]}"
elif VIDEO_SOURCE == "Use Video from Google Drive":
    resolved_video = VIDEO_CUSTOM_PATH.strip()
    if not Path(resolved_video).exists():
        drive_vids = list(Path("/content/drive/MyDrive").glob("**/*.mp4"))
        if drive_vids:
            resolved_video = str(drive_vids[0])
            print("[FOUND] Found video in Google Drive:", resolved_video)
        else:
            raise FileNotFoundError(f"Video file not found: {VIDEO_CUSTOM_PATH}")
else:
    # Compile Dataset Validation Frames to Video
    dataset_path = Path("/content/yolo_master/dataset")
    val_imgs = sorted(list(dataset_path.glob("**/val/**/*.jpg")) + list(dataset_path.glob("**/val/**/*.png")))
    if not val_imgs:
        val_imgs = sorted(list(dataset_path.glob("**/*.jpg")) + list(dataset_path.glob("**/*.png")))
    if not val_imgs:
        raise FileNotFoundError("No dataset images found in /content/yolo_master/dataset to compile video!")
    print(f"Compiling {len(val_imgs)} dataset frames into MP4 video stream...")
    resolved_video = "/content/compiled_dataset_stream.mp4"
    first_img = cv2.imread(str(val_imgs[0]))
    vh, vw = first_img.shape[:2]
    vwtr = cv2.VideoWriter(resolved_video, cv2.VideoWriter_fourcc(*'mp4v'), 15.0, (vw, vh))
    for im_p in val_imgs:
        im = cv2.imread(str(im_p))
        if im is not None:
            if (im.shape[1], im.shape[0]) != (vw, vh):
                im = cv2.resize(im, (vw, vh))
            vwtr.write(im)
    vwtr.release()
    print("[OK] Compiled video created successfully:", resolved_video)

print("[OK] All inputs validated successfully!")


In [ ]:
#@title 5. Execute Video Inference & Side-by-Side Comparison
import time
import json
import shutil
import subprocess
import numpy as np
import cv2
from ultralytics import YOLO

OUTPUT_RAW_VIDEO = "/content/SVG_comparison_raw.mp4"
OUTPUT_FINAL_VIDEO = "/content/SVG_Model_Comparison_SideBySide.mp4"
OUTPUT_REPORT_JSON = "/content/SVG_Model_Comparison_Report.json"

def calc_iou(boxA, boxB):
    xA, yA = max(boxA[0], boxB[0]), max(boxA[1], boxB[1])
    xB, yB = min(boxA[2], boxB[2]), min(boxA[3], boxB[3])
    inter = max(0.0, xB - xA) * max(0.0, yB - yA)
    areaA = max(0.0, (boxA[2] - boxA[0]) * (boxA[3] - boxA[1]))
    areaB = max(0.0, (boxB[2] - boxB[0]) * (boxB[3] - boxB[1]))
    union = areaA + areaB - inter
    return float(inter / union) if union > 0 else 0.0

print("Loading models into GPU memory...")
m_new = YOLO(resolved_new_model)
m_old = YOLO(resolved_prev_model) if "Dual" in VERIFICATION_MODE else None
is_dual = ("Dual" in VERIFICATION_MODE and m_old is not None)

cap = cv2.VideoCapture(resolved_video)
if not cap.isOpened():
    raise RuntimeError(f"Cannot open video source: {resolved_video}")

tot_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
v_fps = cap.get(cv2.CAP_PROP_FPS) or 20.0
fw = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
fh = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
limit_frames = min(tot_frames, MAX_FRAMES_TO_PROCESS) if MAX_FRAMES_TO_PROCESS > 0 else tot_frames

# Dimensions: Cap combined width at 1920 for smooth web streaming
pane_w, pane_h = fw, fh
if is_dual and RENDER_MODE == "side_by_side":
    if (fw * 2) > 1920:
        scale = 960.0 / fw
        pane_w, pane_h = int(fw * scale), int(fh * scale)
    out_w, out_h = pane_w * 2, pane_h
else:
    if fw > 1920:
        scale = 1920.0 / fw
        pane_w, pane_h = int(fw * scale), int(fh * scale)
    out_w, out_h = pane_w, pane_h

writer = cv2.VideoWriter(OUTPUT_RAW_VIDEO, cv2.VideoWriter_fourcc(*'mp4v'), v_fps, (out_w, out_h))

# Benchmark tracking
f_idx = 0
stats_old_cnt = 0
stats_new_cnt = 0
stats_old_conf = 0.0
stats_new_conf = 0.0
stats_agreements = 0
agreement_ious = []
new_unique_cnt = 0
t_start = time.time()

name_old = Path(resolved_prev_model or '').stem
name_new = Path(resolved_new_model).stem

print(f"Processing {limit_frames} frames ({out_w}x{out_h} @ {v_fps:.1f} FPS)...\n")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    f_idx += 1
    if MAX_FRAMES_TO_PROCESS > 0 and f_idx > MAX_FRAMES_TO_PROCESS:
        break

    # Inference New Model
    r_new = m_new.predict(frame, imgsz=640, conf=CONFIDENCE_THRESHOLD, device=0, verbose=False)
    boxes_new = []
    if r_new and len(r_new) > 0 and r_new[0].boxes is not None:
        for b in r_new[0].boxes.data.cpu().numpy():
            boxes_new.append({"box": [float(b[0]), float(b[1]), float(b[2]), float(b[3])], "conf": float(b[4]), "cls": int(b[5]), "matched": False})
    stats_new_cnt += len(boxes_new)
    stats_new_conf += sum(b["conf"] for b in boxes_new)

    # Inference Old Model
    boxes_old = []
    if is_dual:
        r_old = m_old.predict(frame, imgsz=640, conf=CONFIDENCE_THRESHOLD, device=0, verbose=False)
        if r_old and len(r_old) > 0 and r_old[0].boxes is not None:
            for b in r_old[0].boxes.data.cpu().numpy():
                boxes_old.append({"box": [float(b[0]), float(b[1]), float(b[2]), float(b[3])], "conf": float(b[4]), "cls": int(b[5])})
        stats_old_cnt += len(boxes_old)
        stats_old_conf += sum(b["conf"] for b in boxes_old)

        # Agreement / IoU calculation
        for nb in boxes_new:
            best_iou = 0.0
            for ob in boxes_old:
                iv = calc_iou(nb["box"], ob["box"])
                if iv > best_iou:
                    best_iou = iv
            if best_iou >= IOU_MATCH_THRESHOLD:
                nb["matched"] = True
                stats_agreements += 1
                agreement_ious.append(best_iou)
            else:
                new_unique_cnt += 1

    # Rendering
    if is_dual and RENDER_MODE == "side_by_side":
        # Left Frame: Previous Model (Orange)
        f_left = frame.copy()
        for ob in boxes_old:
            ox1, oy1, ox2, oy2 = map(int, ob["box"])
            cv2.rectangle(f_left, (ox1, oy1), (ox2, oy2), (245, 158, 11), 2)
            lbl = f"OLD {int(ob['conf']*100)}%"
            (tw, th), _ = cv2.getTextSize(lbl, cv2.FONT_HERSHEY_SIMPLEX, 0.45, 1)
            cv2.rectangle(f_left, (ox1, oy1 - th - 6), (ox1 + tw + 6, oy1), (245, 158, 11), -1)
            cv2.putText(f_left, lbl, (ox1 + 3, oy1 - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (15, 23, 42), 1, cv2.LINE_AA)
        cv2.rectangle(f_left, (0, 0), (fw, 30), (15, 23, 42), -1)
        cv2.putText(f_left, f"PREVIOUS: {name_old.upper()}", (14, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.48, (245, 158, 11), 1, cv2.LINE_AA)

        # Right Frame: New Model (Cyan)
        f_right = frame.copy()
        for nb in boxes_new:
            nx1, ny1, nx2, ny2 = map(int, nb["box"])
            bcol = (56, 189, 248) if nb["matched"] else (52, 211, 153)
            cv2.rectangle(f_right, (nx1, ny1), (nx2, ny2), bcol, 2)
            tag = f"NEW {int(nb['conf']*100)}%" if nb["matched"] else f"NEW+ {int(nb['conf']*100)}%"
            (tw, th), _ = cv2.getTextSize(tag, cv2.FONT_HERSHEY_SIMPLEX, 0.45, 1)
            cv2.rectangle(f_right, (nx1, ny1 - th - 6), (nx1 + tw + 6, ny1), bcol, -1)
            cv2.putText(f_right, tag, (nx1 + 3, ny1 - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (15, 23, 42), 1, cv2.LINE_AA)
        cv2.rectangle(f_right, (0, 0), (fw, 30), (15, 23, 42), -1)
        cv2.putText(f_right, f"NEW (BEST.PT): {name_new.upper()}", (14, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.48, (56, 189, 248), 1, cv2.LINE_AA)

        if (pane_w, pane_h) != (fw, fh):
            f_left = cv2.resize(f_left, (pane_w, pane_h), interpolation=cv2.INTER_AREA)
            f_right = cv2.resize(f_right, (pane_w, pane_h), interpolation=cv2.INTER_AREA)
        combined = np.hstack([f_left, f_right])
        cv2.line(combined, (pane_w, 0), (pane_w, pane_h), (51, 65, 85), 2)

        # Master Telemetry HUD
        cv2.rectangle(combined, (0, 0), (out_w, 56), (15, 23, 42), -1)
        cv2.putText(combined, f"OLD: {len(boxes_old)} dets", (16, 36), cv2.FONT_HERSHEY_SIMPLEX, 0.52, (245, 158, 11), 1, cv2.LINE_AA)
        cv2.putText(combined, f"FRAME {f_idx}/{limit_frames}", (pane_w - 70, 36), cv2.FONT_HERSHEY_SIMPLEX, 0.52, (255, 255, 255), 1, cv2.LINE_AA)
        cv2.putText(combined, f"NEW: {len(boxes_new)} dets | Agreement: {stats_agreements}", (pane_w + 16, 36), cv2.FONT_HERSHEY_SIMPLEX, 0.52, (56, 189, 248), 1, cv2.LINE_AA)
        writer.write(combined)
    else:
        # Single Model View
        f_out = frame.copy()
        for nb in boxes_new:
            nx1, ny1, nx2, ny2 = map(int, nb["box"])
            cv2.rectangle(f_out, (nx1, ny1), (nx2, ny2), (56, 189, 248), 2)
            lbl = f"package {int(nb['conf']*100)}%"
            cv2.putText(f_out, lbl, (nx1, ny1 - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (56, 189, 248), 1, cv2.LINE_AA)
        if (pane_w, pane_h) != (fw, fh):
            f_out = cv2.resize(f_out, (pane_w, pane_h), interpolation=cv2.INTER_AREA)
        cv2.rectangle(f_out, (0, 0), (out_w, 48), (15, 23, 42), -1)
        cv2.putText(f_out, f"VISIONFORGE YOLO BEST.PT | FRAME {f_idx}/{limit_frames} | DETECTIONS: {len(boxes_new)}", (16, 32), cv2.FONT_HERSHEY_SIMPLEX, 0.52, (56, 189, 248), 1, cv2.LINE_AA)
        writer.write(f_out)

    if f_idx % 30 == 0 or f_idx == limit_frames:
        pct = (f_idx / limit_frames) * 100
        print(f"Frame {f_idx:4d}/{limit_frames} ({pct:5.1f}%) | New Dets: {len(boxes_new)} | Agreements: {stats_agreements}")

cap.release()
writer.release()

# Transcode with ffmpeg for browser compatibility
print("\nTranscoding to H.264 MP4 for instant browser streaming...")
subprocess.run([
    "ffmpeg", "-y", "-i", OUTPUT_RAW_VIDEO, "-c:v", "libx264", "-pix_fmt", "yuv420p",
    "-crf", "23", "-preset", "fast", OUTPUT_FINAL_VIDEO
], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Copy to Google Drive
if DRIVE_ROOT.exists():
    shutil.copy2(OUTPUT_FINAL_VIDEO, DRIVE_ROOT / "SVG_Model_Comparison_SideBySide.mp4")

# Generate Report
avg_old = (stats_old_conf / max(1, stats_old_cnt))
avg_new = (stats_new_conf / max(1, stats_new_cnt))
report_data = {
    "total_frames_evaluated": f_idx,
    "previous_model": {"name": name_old, "detections": stats_old_cnt, "avg_confidence": round(avg_old, 4)},
    "new_model": {"name": name_new, "detections": stats_new_cnt, "avg_confidence": round(avg_new, 4)},
    "mutual_agreements": stats_agreements,
    "unique_new_detections": new_unique_cnt,
    "mean_agreement_iou": round(float(np.mean(agreement_ious)), 4) if agreement_ious else 0.0,
    "confidence_delta_pct": round((avg_new - avg_old) * 100, 2)
}
Path(OUTPUT_REPORT_JSON).write_text(json.dumps(report_data, indent=2))
if DRIVE_ROOT.exists():
    shutil.copy2(OUTPUT_REPORT_JSON, DRIVE_ROOT / "SVG_Model_Comparison_Report.json")

print("=" * 65)
print("             MODEL COMPARISON BENCHMARK SUMMARY")
print("=" * 65)
print(f"Total Frames Evaluated:       {f_idx}")
if is_dual:
    print(f"Previous Model Detections:   {stats_old_cnt} (Avg Conf: {avg_old*100:.1f}%)")
    print(f"New Model (best.pt) Dets:    {stats_new_cnt} (Avg Conf: {avg_new*100:.1f}%)")
    print(f"Mutual Agreements:           {stats_agreements} (Mean Overlap IoU: {report_data['mean_agreement_iou']*100:.1f}%)")
    print(f"New Unique Detections:       {new_unique_cnt}")
    print(f"Confidence Gain:             {report_data['confidence_delta_pct']:+.2f}%")
else:
    print(f"New Model (best.pt) Dets:    {stats_new_cnt} (Avg Conf: {avg_new*100:.1f}%)")
print(f"Annotated Video Output:      {OUTPUT_FINAL_VIDEO}")
print(f"Google Drive Checkpoint:     {DRIVE_ROOT / 'SVG_Model_Comparison_SideBySide.mp4'}")
print("=" * 65)


In [ ]:
#@title 6. Watch Annotated Video in Colab & Download to PC
import base64
from IPython.display import HTML, display
from google.colab import files

DOWNLOAD_TO_PC = True #@param {type:"boolean"}

# Display HTML5 video player
vid_p = Path(OUTPUT_FINAL_VIDEO)
if vid_p.exists() and vid_p.stat().st_size < 80 * 1024 * 1024:
    with open(vid_p, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    html = f'''
    <div style="background: #0f172a; padding: 16px; border-radius: 12px; border: 1px solid #334155; margin-bottom: 12px;">
      <h4 style="color: #38bdf8; margin: 0 0 10px 0; font-family: sans-serif;">▶ VisionForge Video Player (Synchronized Comparison)</h4>
      <video width="100%" style="border-radius: 8px; max-height: 520px; outline: none; background: #000;" controls autoplay loop>
        <source src="data:video/mp4;base64,{b64}" type="video/mp4">
        Your browser does not support HTML5 video.
      </video>
    </div>
    '''
    display(HTML(html))
else:
    print("Video file ready:", OUTPUT_FINAL_VIDEO)

if DOWNLOAD_TO_PC:
    print("Downloading video and metrics report to your computer...")
    files.download(OUTPUT_FINAL_VIDEO)
    files.download(OUTPUT_REPORT_JSON)
